In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import re


def parse_result_config(name: str) -> tuple[int, str] | None:
    r"""
    Parse a result configuration name to recover layer index and module name.

    Parameters
    ----------
    name: str
        Result configuration name of the form "layer_0_module_act".

    Returns
    -------
    tuple[int, str]
        Tuple containing model name, layer index and module name.
    """
    pattern = re.compile(
        r"^"
        r"layer_"
        r"(\d+)"  # layer
        r"_module_"
        r"(.+)"  # module name
        r".json"
    )
    match = pattern.match(name)
    if not match:
        return None

    layer, module = match.groups()

    return int(layer), module


parse_result_config("layer_0_module_act.json")


(0, 'act')

In [1]:
l = "test"
l = sorted(l)
print(l)
print("".join(l))

['e', 's', 't', 't']
estt


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the evaluation matrix
model_pairs = {
    "Gemma 3 4B (Base)": "google/gemma-3-4b-pt",
    "Gemma 3 4B (Instruct)": "google/gemma-3-4b-it",
    "Gemma 4 E2B (Base)": "google/gemma-4-E2B",
    "Gemma 4 E2B (Instruct)": "google/gemma-4-E2B-it",
}

# Standard text completion prompt for base testing
raw_prompt = "The primary objective of a linear probing experiment is"

# A reasoning/riddle context that forces the Instruct models to trigger thinking blocks
chat_messages = [
    {
        "role": "user",
        "content": "Solve this riddle: A man pushes his car to a hotel and tells the owner he's bankrupt. Why?",
    }
]

for name, model_id in model_pairs.items():
    print("\n" + "=" * 60)
    print(f" Testing: {name}")
    print("=" * 60)

    # 1. Load Tokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
    except Exception as e:
        print(f"[-] Could not load tokenizer for {model_id}: {e}")
        continue

    # 2. Inspect Chat Template to confirm variant posture
    is_instruct = "Instruct" in name
    if tokenizer.chat_template is not None:
        print(f"[Template Status]: Active. Special tokens mapped: {list(tokenizer.special_tokens_map.keys())}")
    else:
        print("[Template Status]: Inactive / None (Standard Base Text Completer).")

    # 3. Format Input Sequence Based on Variant Design
    if is_instruct:
        print("[Pipeline Setup]: Formatting via official chat template to trigger instruction features...")
        formatted_input = tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)
    else:
        print("[Pipeline Setup]: Passing clean raw prompt text directly...")
        formatted_input = raw_prompt

    # 4. Run Inference Validation
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

        inputs = tokenizer(formatted_input, return_tensors="pt").to(model.device)

        print(f"Input Token IDs: {inputs['input_ids'].tolist()[0]}")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,  # Increase budget to fit thinking + final answer
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                # Force the model architecture to allocate thinking iterations
                min_thinking_tokens=32,
                thinking_budget=1024,
            )

        generated_ids = outputs[0].tolist()
        new_ids = generated_ids[len(inputs["input_ids"][0]) :]

        print(f"Generated Token IDs: {new_ids}")
        print(f"Decoded Output Text:\n{'-' * 30}\n{tokenizer.decode(new_ids)}\n{'-' * 30}")

        # 5. Audit the generated sequence for any non-standard/control tokens
        # We define structural tokens as any special ID that is NOT pad, bos, or eos.
        standard_ids = {tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id}
        found_hidden_control_tokens = False
        captured_control_tokens = []

        for token_id in new_ids:
            if token_id in tokenizer.all_special_ids and token_id not in standard_ids:
                found_hidden_control_tokens = True
                captured_control_tokens.append(tokenizer.convert_ids_to_tokens(token_id))

        print(f"Contains non-standard control tokens (Turn/Think tags)? {found_hidden_control_tokens}")
        if found_hidden_control_tokens:
            print(f" -> Hidden Control Tokens Detected: {list(set(captured_control_tokens))}")

    except Exception as e:
        print(f"[-] Execution failure on weight checks: {e}")



 Testing: Gemma 3 4B (Base)
[Template Status]: Inactive / None (Standard Base Text Completer).
[Pipeline Setup]: Passing clean raw prompt text directly...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input Token IDs: [2, 818, 5905, 12092, 529, 496, 6373, 83454, 5274, 563]
[-] Execution failure on weight checks: The following `model_kwargs` are not used by the model: ['min_thinking_tokens', 'thinking_budget'] (note: typos in the generate arguments will also show up in this list)

 Testing: Gemma 3 4B (Instruct)
[Template Status]: Active. Special tokens mapped: ['bos_token', 'eos_token', 'unk_token', 'pad_token', 'mask_token', 'boi_token', 'eoi_token', 'image_token']
[Pipeline Setup]: Formatting via official chat template to trigger instruction features...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Input Token IDs: [2, 2, 105, 2364, 107, 76857, 672, 201592, 236787, 562, 880, 57315, 914, 1295, 531, 496, 8217, 532, 12630, 506, 8123, 668, 236789, 236751, 67311, 236761, 8922, 236881, 106, 107, 105, 4368, 107]
[-] Execution failure on weight checks: The following `model_kwargs` are not used by the model: ['min_thinking_tokens', 'thinking_budget'] (note: typos in the generate arguments will also show up in this list)

 Testing: Gemma 4 E2B (Base)
[Template Status]: Inactive / None (Standard Base Text Completer).
[Pipeline Setup]: Passing clean raw prompt text directly...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Input Token IDs: [2, 818, 5905, 12092, 529, 496, 6373, 83454, 5274, 563]
[-] Execution failure on weight checks: The following `model_kwargs` are not used by the model: ['min_thinking_tokens', 'thinking_budget'] (note: typos in the generate arguments will also show up in this list)

 Testing: Gemma 4 E2B (Instruct)
[Template Status]: Active. Special tokens mapped: ['bos_token', 'eos_token', 'unk_token', 'pad_token', 'mask_token', 'audio_token', 'boa_token', 'boi_token', 'eoa_token', 'eoc_token', 'eoi_token', 'eot_token', 'escape_token', 'etc_token', 'etd_token', 'etr_token', 'image_token', 'soc_token', 'sot_token', 'stc_token', 'std_token', 'str_token', 'think_token']
[Pipeline Setup]: Formatting via official chat template to trigger instruction features...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Input Token IDs: [2, 105, 2364, 107, 76857, 672, 201592, 236787, 562, 880, 57315, 914, 1295, 531, 496, 8217, 532, 12630, 506, 8123, 668, 236789, 236751, 67311, 236761, 8922, 236881, 106, 107, 105, 4368, 107]
[-] Execution failure on weight checks: The following `model_kwargs` are not used by the model: ['min_thinking_tokens', 'thinking_budget'] (note: typos in the generate arguments will also show up in this list)


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the pairs of models you want to test
model_pairs = {
    "Gemma 3 4B (Base)": "google/gemma-3-4b-pt",
    "Gemma 3 4B (Instruct)": "google/gemma-3-4b-it",
    "Gemma 4 E2B (Base)": "google/gemma-4-E2B",
    "Gemma 4 E2B (Instruct)": "google/gemma-4-E2B-it",
}

sample_prompt = "The primary objective of a linear probing experiment is"

for name, model_id in model_pairs.items():
    print(f"\n{'=' * 10} Testing: {name} {'=' * 10}")

    # 1. Load the Tokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
    except Exception as e:
        print(f"Could not load tokenizer for {model_id}: {e}")
        continue

    # 2. Check the Chat Template Status
    if tokenizer.chat_template is not None:
        print("[Template Status]: Instruct/Chat Template is ACTIVE.")
        # Print a few identified special control strings if they exist
        print(f" -> Special tokens in vocab: {list(tokenizer.special_tokens_map.keys())}")
    else:
        print("[Template Status]: No explicit chat template found (Standard Base Model behavior).")

    # 3. Load the Model and Test Output Production
    try:
        # Load in 16-bit or 8-bit to optimize memory overhead on your cluster
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

        inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)

        # Track input token ids
        print(f"Input Token IDs: {inputs['input_ids'].tolist()[0]}")

        # Generate completion tokens
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)

        generated_ids = outputs[0].tolist()
        new_ids = generated_ids[len(inputs["input_ids"][0]) :]

        print(f"Generated Token IDs: {new_ids}")
        print(f"Decoded Output Text: '{tokenizer.decode(new_ids)}'")

        # Check if any special tokens crept into the generation stream
        has_special = any(idx in tokenizer.all_special_ids for idx in new_ids)
        print(f"Contains hidden/special control tokens? {has_special}")

    except Exception as e:
        print(f"Skipping weight generation check: {e}")
        print("(Ensure you are logged into Hugging Face and have weight access permissions).")



========== Testing: Gemma 3 4B (Base) ==========
[Template Status]: No explicit chat template found (Standard Base Model behavior).


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input Token IDs: [2, 818, 5905, 12092, 529, 496, 6373, 83454, 5274, 563]
Generated Token IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Decoded Output Text: '<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>'
Contains hidden/special control tokens? True

========== Testing: Gemma 3 4B (Instruct) ==========
[Template Status]: Instruct/Chat Template is ACTIVE.
 -> Special tokens in vocab: ['bos_token', 'eos_token', 'unk_token', 'pad_token', 'mask_token', 'boi_token', 'eoi_token', 'image_token']


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

... (9 outputs left out)

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Input Token IDs: [818, 5905, 12092, 529, 496, 6373, 83454, 5274, 563]
Generated Token IDs: [496, 6373, 83454, 5274, 563, 496, 6373, 83454, 5274, 563, 496, 6373, 83454, 5274, 563, 496, 6373, 83454, 5274, 563]
Decoded Output Text: ' a linear probing experiment is a linear probing experiment is a linear probing experiment is a linear probing experiment is'
Contains hidden/special control tokens? False


In [5]:
from transformers import AutoConfig

model_name = "meta-llama/Llama-3.2-3B-Instruct"
# Fetch the configuration from the Hugging Face hub
config = AutoConfig.from_pretrained(model_name)

# Print the number of layers
print(f"Number of layers: {config.num_hidden_layers}")
print(f"Hidden size (width): {config.hidden_size}")


Number of layers: 28
Hidden size (width): 3072


In [1]:
from transformers import AutoConfig

model_name = "google/gemma-3-12b-pt"
# Fetch the configuration from the Hugging Face hub
text_config = AutoConfig.from_pretrained(model_name).text_config

# Print the number of layers
print(f"Number of layers: {text_config.num_hidden_layers}")
print(f"Hidden size (width): {text_config.hidden_size}")


Number of layers: 48
Hidden size (width): 3840


In [ ]:
import time

import mteb
from sentence_transformers import SentenceTransformer

device = "cuda:6"
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)

task = mteb.get_task("ItaCaseholdClassification")

# Test without cache first
start = time.time()
results = mteb.evaluate(model, tasks=task)
print(f"No cache: {time.time() - start:.2f}s")

# Then test with cache
from mteb.cache import ResultCache

cache = ResultCache(cache_path="~/.cache/mteb/test")
start = time.time()
results = mteb.evaluate(
    model,
    tasks=task,
    cache=cache,
)
print(f"With cache: {time.time() - start:.2f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/mteb/models/model_meta.py:764: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/mteb/models/model_meta.py:726: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),
/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precisi

No cache: 29.36s


/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

With cache: 30.84s


/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/data_1/aodonnat/miniforge3/envs/test/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [4]:
import mteb
from sentence_transformers import SentenceTransformer

device = "cuda:6"
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)
nb_params = sum(p.numel() for p in model.parameters())
print(f"{nb_params:,}")
task = mteb.get_task("ItaCaseholdClassification")
cache = mteb.ResultCache(cache_path="~/.cache/mteb/neww")
results = mteb.evaluate(model, tasks=task, cache=cache, verbosity=3)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


22,713,216


TypeError: evaluate() got an unexpected keyword argument 'verbosity'

In [2]:
from transformers import AutoModel

model_name = "google/gemma-3-1b-pt"
model = AutoModel.from_pretrained(model_name)
print(f"✓ Successfully loaded model: {model_name}")


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✓ Successfully loaded model: google/gemma-3-1b-pt


In [3]:
from transformers import AutoTokenizer

model_name = "google/gemma-3-1b-pt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"✓ Successfully loaded tokenizer: {model_name}")


✓ Successfully loaded tokenizer: google/gemma-3-1b-pt


In [4]:
from transformers import AutoTokenizer

model_name = "lgrobol/roberta-minuscule"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"✓ Successfully loaded tokenizer: {model_name}")


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✓ Successfully loaded tokenizer: lgrobol/roberta-minuscule


In [2]:
import os
from pathlib import Path

cache_dir = Path.home() / ".cache" / "huggingface" / "hub"

# List cached models
for model_path in cache_dir.glob("models--*"):
    print(f"✓ Cached: {model_path.name}")

# Check specifically for gemma
gemma_path = cache_dir / "models--google--gemma-3-1b-pt"
if gemma_path.exists():
    print(f"✓ Gemma is cached at: {gemma_path}")
else:
    print("✗ Gemma not found in cache")


✓ Cached: models--facebook--dinov3-vith16plus-pretrain-lvd1689m
✓ Cached: models--facebook--vit-mae-large
✓ Cached: models--intfloat--multilingual-e5-large
✓ Cached: models--Salesforce--SFR-Embedding-Mistral
✓ Cached: models--facebook--vit-mae-base
✓ Cached: models--google--gemma-3-4b-pt
✓ Cached: models--google--siglip2-base-patch16-224
✓ Cached: models--openai-community--gpt2-xl
✓ Cached: models--apple--aimv2-3B-patch14-224
✓ Cached: models--meta-llama--Llama-3.2-1B
✓ Cached: models--Qwen--Qwen3-32B
✓ Cached: models--google--gemma-3-27b-pt
✓ Cached: models--facebook--dinov3-vits16-pretrain-lvd1689m
✓ Cached: models--facebook--dinov3-vitb16-pretrain-lvd1689m
✓ Cached: models--Qwen--Qwen3-Embedding-8B
✓ Cached: models--intfloat--multilingual-e5-small
✓ Cached: models--google--gemma-4-E4B
✓ Cached: models--intfloat--e5-mistral-7b-instruct
✓ Cached: models--facebook--dinov3-vitl16-pretrain-lvd1689m
✓ Cached: models--Qwen--Qwen3-1.7B
✓ Cached: models--google--vit-base-patch16-224
✓ Cached

In [ ]:
from pathlib import Path

from transformers import AutoTokenizer

# Point to the snapshots folder (not blobs)
cache_root = Path.home() / ".cache" / "huggingface" / "hub" / "models--google--gemma-3-1b-pt"
snapshots_dir = cache_root / "snapshots"

# List available snapshots
snapshots = [d for d in snapshots_dir.iterdir() if d.is_dir()]
print(f"Available snapshots: {snapshots}")

if snapshots:
    # Use the first (and usually only) snapshot folder
    model_path = snapshots[0]
    print(f"Loading from: {model_path}")

    # Now this works - model_path is a directory, not a blob file
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    print("✓ Tokenizer loaded!")


Available snapshots: [PosixPath('/mnt/data_1/aodonnat/.cache/huggingface/hub/models--google--gemma-3-1b-pt/snapshots/fcf18a2a879aab110ca39f8bffbccd5d49d8eb29')]
Loading from: /mnt/data_1/aodonnat/.cache/huggingface/hub/models--google--gemma-3-1b-pt/snapshots/fcf18a2a879aab110ca39f8bffbccd5d49d8eb29
✓ Tokenizer loaded!


In [ ]:
# Test basic connectivity
import socket

import requests
from huggingface_hub import HfApi

# Check current proxy settings
print("HTTP_PROXY:", os.environ.get("HTTP_PROXY", "Not set"))
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY", "Not set"))
print("http_proxy:", os.environ.get("http_proxy", "Not set"))
print("https_proxy:", os.environ.get("https_proxy", "Not set"))
print("NO_PROXY:", os.environ.get("NO_PROXY", "Not set"))
print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT", "Not set"))

# Test direct connection to HuggingFace
try:
    api = HfApi()
    models = api.list_models(search="all-MiniLM-L6-v2", limit=1)
    print("✓ HuggingFace Hub API accessible")
except Exception as e:
    print(f"✗ HuggingFace Hub API error: {e}")

# Test socket connection
try:
    sock = socket.create_connection(("huggingface.co", 443), timeout=5)
    sock.close()
    print("✓ Direct connection to huggingface.co successful")
except Exception as e:
    print(f"✗ Direct connection failed: {e}")


HTTP_PROXY: Not set
HTTPS_PROXY: Not set
http_proxy: http://127.0.0.1:3128
https_proxy: http://127.0.0.1:3128
NO_PROXY: Not set
HF_ENDPOINT: Not set
✓ HuggingFace Hub API accessible
✗ Direct connection failed: [Errno 101] Network is unreachable


In [ ]:
import os
import socket

# Test if requests library can use the proxy
print("Testing requests library with proxy...")
try:
    # This should use the proxy automatically
    response = requests.get("https://huggingface.co", timeout=10)
    print(f"✓ requests library: Status {response.status_code}")
except Exception as e:
    print(f"✗ requests library failed: {e}")

# Test DNS resolution
print("\nTesting DNS resolution...")
try:
    ip = socket.gethostbyname("huggingface.co")
    print(f"✓ DNS resolution: huggingface.co -> {ip}")
except Exception as e:
    print(f"✗ DNS resolution failed: {e}")

# Check if uppercase proxy variables help
print("\nCurrent proxy environment variables:")
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    print(f"  {var}: {os.environ.get(var, 'Not set')}")

# Test with explicit proxy
print("\nTesting with explicit proxy...")
try:
    session = requests.Session()
    session.proxies = {"http": "http://127.0.0.1:3128", "https": "http://127.0.0.1:3128"}
    response = session.get("https://huggingface.co/api/models?search=all-MiniLM", timeout=10)
    print(f"✓ Explicit proxy works: Found {len(response.json())} models")
except Exception as e:
    print(f"✗ Explicit proxy failed: {e}")


Testing requests library with proxy...
✗ requests library failed: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)

Testing DNS resolution...
✓ DNS resolution: huggingface.co -> 98.159.108.71

Current proxy environment variables:
  HTTP_PROXY: Not set
  HTTPS_PROXY: Not set
  http_proxy: http://127.0.0.1:3128
  https_proxy: http://127.0.0.1:3128

Testing with explicit proxy...
✗ Explicit proxy failed: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)


In [ ]:
import os

# Set proxy before anything
os.environ["http_proxy"] = "http://127.0.0.1:3128"
os.environ["https_proxy"] = "http://127.0.0.1:3128"
os.environ["HTTP_PROXY"] = "http://127.0.0.1:3128"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:3128"

# Check what's actually being used
import requests

print("Requests version:", requests.__version__)

# Check if proxy is being picked up
session = requests.Session()
print("Default session proxies:", session.proxies)

# Test direct API call (this worked before)
from huggingface_hub import HfApi

api = HfApi()
try:
    models = api.list_models(search="all-MiniLM-L6-v2", limit=1)
    print("✓ API call works")
except Exception as e:
    print(f"✗ API call failed: {e}")

# Now test file download specifically
from huggingface_hub import hf_hub_download

try:
    # Try to download just the config file (small)
    path = hf_hub_download(
        repo_id="sentence-transformers/all-MiniLM-L6-v2", filename="config.json", local_files_only=False
    )
    print(f"✓ Download works! File at: {path}")
except Exception as e:
    print(f"✗ Download failed: {e}")
    print(f"Error type: {type(e).__name__}")


Requests version: 2.32.5
Default session proxies: {'http': 'http://127.0.0.1:3128', 'https': 'http://127.0.0.1:3128'}
✓ API call works
✓ Download works! File at: /mnt/data_1/aodonnat/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json


In [1]:
from transformers import AutoModel

model_name = "google/gemma-3-1b-pt"
model = AutoModel.from_pretrained(model_name)
print(f"✓ Successfully loaded model: {model_name}")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✓ Successfully loaded model: google/gemma-3-1b-pt


In [4]:
from transformers import Gemma3TextModel

model_name = "google/gemma-3-1b-pt"
model = Gemma3TextModel.from_pretrained(model_name, device_map="cuda:4")
print(f"✓ Successfully loaded model: {model_name}")


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✓ Successfully loaded model: google/gemma-3-1b-pt


In [ ]:
import torch
from transformers import Gemma3TextModel

model_name = "google/gemma-3-1b-pt"
dtype = torch.bfloat16
device = "cuda:4"
model = Gemma3TextModel.from_pretrained(
    model_name,
    dtype=dtype,
    device_map=device,
)
model.eval()
print(f"✓ Successfully loaded model: {model_name}")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✓ Successfully loaded model: google/gemma-3-1b-pt


In [ ]:
from waldo.models import Gemma3, Gemma3Config

model_name = "gemma3"
model_size = "1b"
device = "cuda:4"
config = Gemma3Config(model_name=model_name, model_size=model_size, device=device)
model = Gemma3(config=config)
print(f"✓ Successfully loaded model: {model_name}-{model_size} on {device}")

load model: google/gemma-3-1b-pt


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✓ Successfully loaded model: google/gemma-3-1b-pt
load model: google/gemma-3-1b-pt


In [1]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")


In [1]:
from waldo.models.utils import build_model

model_config = {
    "model_name": "gemma3",
    "model_size": "1b",
    "device": "cuda:4",
}
model = build_model(config=model_config, load_offline=True, return_config=False)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [ ]:
import signal

from huggingface_hub import hf_hub_download

signal.alarm(10)  # 10 second timeout
try:
    path = hf_hub_download(repo_id="sentence-transformers/all-MiniLM-L6-v2", filename="config.json")
    print(f"✓ {path}")
except Exception as e:
    print(f"✗ {e}")
finally:
    signal.alarm(0)


: 

In [3]:
import httpx

try:
    # Hugging Face Hub uses httpx under the hood
    response = httpx.get("https://huggingface.co/api/whoami-v2", timeout=5.0)
    print(f"Success! Response code: {response.status_code}")
except Exception as e:
    print(f"Connection failed: {e}")


Connection failed: timed out


In [ ]:
import mteb
from sentence_transformers import SentenceTransformer

device = "cuda:6"
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)
nb_params = sum(p.numel() for p in model.parameters())
print(f"{nb_params:,}")
task = mteb.get_task("ItaCaseholdClassification")
cache = mteb.ResultCache(cache_path="~/.cache/mteb/new")
results = mteb.evaluate(model, tasks=task, cache=cache)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


22,713,216


/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [1]:
from waldo.data import build_mteb

config = {"benchmark_name": "Multilingual", "task_type": "Classification", "max_n_docs": 8192}
benchmark = build_mteb(config)
print(len(benchmark.languages), benchmark.n_tasks, len(benchmark.domains))
benchmark.tasks


31 24 14


[BulgarianStoreReviewSentimentClassfication(name='BulgarianStoreReviewSentimentClassfication', languages=['bul']),
 DBpediaClassification(name='DBpediaClassification', languages=['eng']),
 FinancialPhrasebankClassification(name='FinancialPhrasebankClassification', languages=['eng']),
 PoemSentimentClassification(name='PoemSentimentClassification', languages=['eng']),
 TweetTopicSingleClassification(name='TweetTopicSingleClassification', languages=['eng']),
 EstonianValenceClassification(name='EstonianValenceClassification', languages=['est']),
 GujaratiNewsClassification(name='GujaratiNewsClassification', languages=['guj']),
 SentimentAnalysisHindi(name='SentimentAnalysisHindi', languages=['hin']),
 IndonesianIdClickbaitClassification(name='IndonesianIdClickbaitClassification', languages=['ind']),
 ItaCaseholdClassification(name='ItaCaseholdClassification', languages=['ita']),
 KorSarcasmClassification(name='KorSarcasmClassification', languages=['kor']),
 KurdishSentimentClassification

In [ ]:
import mteb
from sentence_transformers import SentenceTransformer
from waldo.config import DEVICE

model_name = (
    "all-MiniLM-L6-v2"  # "intfloat/e5-mistral-7b-instruct"  # "intfloat/multilingual-e5-large"  # "all-MiniLM-L6-v2"
)
model = SentenceTransformer(model_name, device=DEVICE)
nb_params = sum(p.numel() for p in model.parameters())
print(f"{nb_params:,}")


KeyboardInterrupt: 

In [3]:
model = model.to(DEVICE)
task = mteb.get_task("FilipinoShopeeReviewsClassification")
cache = mteb.ResultCache(cache_path="~/.cache/mteb")
results = mteb.evaluate(model, tasks=task, cache=cache)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/202k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/203k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2048 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2048 [00:00<?, ? examples/s]

/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_job

In [4]:
model = model.to(DEVICE)
task = mteb.get_task("BulgarianStoreReviewSentimentClassfication")
cache = mteb.ResultCache(cache_path="~/.cache/mteb")
results = mteb.evaluate(model, tasks=task, cache=cache)


Task name web_search_query is not valid. Valid keys are task types [('BitextMining', 'Classification', 'MultilabelClassification', 'Clustering', 'PairClassification', 'Regression', 'Reranking', 'Retrieval', 'STS', 'Summarization', 'InstructionRetrieval', 'InstructionReranking', 'Any2AnyMultiChoice', 'Any2AnyRetrieval', 'Any2AnyMultilingualRetrieval', 'VisionCentricQA', 'ImageClustering', 'ImageClassification', 'ImageMultilabelClassification', 'DocumentUnderstanding', 'VisualSTS(eng)', 'VisualSTS(multi)', 'ZeroShotClassification', 'Compositionality', 'AudioClustering', 'AudioMultilabelClassification', 'AudioReranking', 'AudioZeroshotClassification', 'AudioClassification', 'AudioCrossFoldClassification', 'AudioPairClassification')], prompt types [['query', 'document']], and task names
Task name sts_query is not valid. Valid keys are task types [('BitextMining', 'Classification', 'MultilabelClassification', 'Clustering', 'PairClassification', 'Regression', 'Reranking', 'Retrieval', 'STS',

In [34]:
model = model.to(DEVICE)
task = mteb.get_task("BulgarianStoreReviewSentimentClassfication")
cache = mteb.ResultCache(cache_path="~/.cache/mteb")
results = mteb.evaluate(model, tasks=task, cache=cache)


/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_job

In [ ]:
model = model.to(DEVICE)
task = mteb.get_task("BulgarianStoreReviewSentimentClassfication")
cache = mteb.ResultCache(cache_path="~/.cache/mteb")
results = mteb.evaluate(model, tasks=task, cache=cache)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/203k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/56.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/724 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/182 [00:00<?, ? examples/s]

/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_job

In [32]:
task = mteb.get_task("GreekLegalCodeClassification")
cache = mteb.ResultCache(cache_path="~/.cache/mteb")
results = mteb.evaluate(model, tasks=task, cache=cache)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.05M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/6.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28536 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2048 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2048 [00:00<?, ? examples/s]

/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/

In [30]:
import mteb

benchmark_name = "Multilingual"
task_type = "Classification"
max_n_docs = 2048

# Get the full English benchmark
benchmark = mteb.get_benchmark(f"MTEB({benchmark_name})")

# Filter to only retrieval tasks
tasks = mteb.filter_tasks(benchmark, task_types=[task_type])
for task in tasks:
    print(task.metadata.name)
# print(
#     len(tasks),
#     tasks[1],
#     tasks[0].metadata.descriptive_stats,
# )
# # Filter out large tasks if max_n_documents is set
# if max_n_docs is not None:
#     tasks = [task for task in tasks if hasattr(task.metadata, "descriptive_stats") and task.metadata.descriptive_stats.get("n_documents", float("inf")) <= max_n_docs]

tasks[0].metadata.descriptive_stats

BulgarianStoreReviewSentimentClassfication
CzechProductReviewSentimentClassification
GreekLegalCodeClassification
DBpediaClassification
FinancialPhrasebankClassification
PoemSentimentClassification
ToxicConversationsClassification
TweetTopicSingleClassification
EstonianValenceClassification
FilipinoShopeeReviewsClassification
GujaratiNewsClassification
SentimentAnalysisHindi
IndonesianIdClickbaitClassification
ItaCaseholdClassification
KorSarcasmClassification
KurdishSentimentClassification
MacedonianTweetSentimentClassification
AfriSentiClassification
AmazonCounterfactualClassification
CataloniaTweetClassification
CyrillicTurkicLangClassification
IndicLangClassification
MasakhaNEWSClassification
MassiveIntentClassification
MultiHateClassification
NordicLangClassification
NusaParagraphEmotionClassification
NusaX-senti
ScalaClassification
SwissJudgementClassification
NepaliNewsClassification
OdiaNewsClassification
PunjabiNewsClassification
PolEmo2.0-OUT
PAC
SinhalaNewsClassification
CSF

{'test': {'num_samples': 182,
  'number_texts_intersect_with_train': 0,
  'text_statistics': {'total_text_length': 57648,
   'min_text_length': 8,
   'average_text_length': 316.74725274725273,
   'max_text_length': 2560,
   'unique_texts': 182},
  'image_statistics': None,
  'label_statistics': {'min_labels_per_text': 1,
   'average_label_per_text': 1.0,
   'max_labels_per_text': 1,
   'unique_labels': 4,
   'labels': {'1': {'count': 118},
    '0': {'count': 33},
    '2': {'count': 25},
    '3': {'count': 6}}}},
 'train': {'num_samples': 724,
  'number_texts_intersect_with_train': None,
  'text_statistics': {'total_text_length': 227740,
   'min_text_length': 10,
   'average_text_length': 314.55801104972375,
   'max_text_length': 1916,
   'unique_texts': 724},
  'image_statistics': None,
  'label_statistics': {'min_labels_per_text': 1,
   'average_label_per_text': 1.0,
   'max_labels_per_text': 1,
   'unique_labels': 4,
   'labels': {'1': {'count': 422},
    '2': {'count': 113},
    '0'

In [13]:
task = mteb.get_task("BulgarianStoreReviewSentimentClassfication")

In [16]:
task.metadata.descriptive_stats


{'test': {'num_samples': 182,
  'number_texts_intersect_with_train': 0,
  'text_statistics': {'total_text_length': 57648,
   'min_text_length': 8,
   'average_text_length': 316.74725274725273,
   'max_text_length': 2560,
   'unique_texts': 182},
  'image_statistics': None,
  'label_statistics': {'min_labels_per_text': 1,
   'average_label_per_text': 1.0,
   'max_labels_per_text': 1,
   'unique_labels': 4,
   'labels': {'1': {'count': 118},
    '0': {'count': 33},
    '2': {'count': 25},
    '3': {'count': 6}}}},
 'train': {'num_samples': 724,
  'number_texts_intersect_with_train': None,
  'text_statistics': {'total_text_length': 227740,
   'min_text_length': 10,
   'average_text_length': 314.55801104972375,
   'max_text_length': 1916,
   'unique_texts': 724},
  'image_statistics': None,
  'label_statistics': {'min_labels_per_text': 1,
   'average_label_per_text': 1.0,
   'max_labels_per_text': 1,
   'unique_labels': 4,
   'labels': {'1': {'count': 422},
    '2': {'count': 113},
    '0'

In [2]:
import types

print(types.UnionType)  # should print <class 'types.UnionType'>


<class 'types.UnionType'>


In [4]:
from waldo.data import build_loader

dataset_name = "food101"
resolution = 224
mode = "static"
transform_type = "static"
num_workers = 8
batch_size = 8
loader_config = {
    "dataset_name": dataset_name,
    "batch_size": batch_size,
    "resolution": resolution,
    "mode": mode,
    "transform_type": transform_type,
}
train_loader = build_loader(config=loader_config, num_workers=num_workers)
for i, batch in enumerate(train_loader):
    x, y = batch
    print(x.shape, y)
    if i > 5:
        break


AssertionError: Invalid mode static. Options are 'train' and 'test.

In [5]:
from waldo.data import build_loader

dataset_name = "aircraft"
resolution = 224
mode = "train"
transform_type = "static"
num_workers = 8
batch_size = 8
loader_config = {
    "dataset_name": dataset_name,
    "batch_size": batch_size,
    "resolution": resolution,
    "mode": mode,
    "transform_type": transform_type,
}
train_loader = build_loader(config=loader_config, num_workers=num_workers)
for i, batch in enumerate(train_loader):
    x, y = batch
    print(x.shape, y)
    if i > 5:
        break

Initializing torchvision.transforms.transforms.Compose | None with value 'Compose(
    Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)' without type checking.


torch.Size([8, 3, 224, 224]) tensor([95, 56, 88, 11, 83, 43, 54,  6])
torch.Size([8, 3, 224, 224]) tensor([97, 44,  8,  4, 34, 79, 39, 49])
torch.Size([8, 3, 224, 224]) tensor([60, 83, 51, 84, 51, 35, 58, 94])
torch.Size([8, 3, 224, 224]) tensor([38, 61, 15, 99, 73, 45, 39, 94])
torch.Size([8, 3, 224, 224]) tensor([72, 26, 87, 68, 72, 79, 51, 99])
torch.Size([8, 3, 224, 224]) tensor([70, 75, 19, 99, 94, 60, 80, 92])
torch.Size([8, 3, 224, 224]) tensor([ 3, 64, 67, 39, 42, 44, 86, 80])


In [ ]:
# import os

# os.environ["OMP_NUM_THREADS"] = "1"

from time import time

from waldo.data import build_loader

dataset_name = "domainnet-clipart"
resolution = 224
mode = "static"
num_workers = 8
batch_size = 512


In [3]:
import os

import psutil

process = psutil.Process(os.getpid())
for num_workers in [8, 2, 4, 8]:
    print(num_workers)
    loader_config = {"dataset_name": dataset_name, "batch_size": batch_size, "resolution": resolution, "mode": mode}
    start = time()
    mem_before = psutil.virtual_memory().available / 1024**3
    train_loader = build_loader(config=loader_config, num_workers=num_workers)
    for i, batch in enumerate(train_loader):
        if i > 5:
            break
    print(time() - start)
    mem_after = psutil.virtual_memory().available / 1024**3
    print(f"Memory used: {mem_before - mem_after:.2f} GB")


8
26.11711883544922
Memory used: 5.67 GB
2
26.554097414016724
Memory used: -3.25 GB
4
16.06286597251892
Memory used: 0.76 GB
8
11.68610405921936
Memory used: -2.13 GB


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = (
    "5"  # important before any import to make sure that MTEB and model loading is only on GPU 5
)

import mteb
from sentence_transformers import SentenceTransformer

DEVICE = "cuda:0"


In [ ]:
model_name = "intfloat/multilingual-e5-small"  # "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=DEVICE)
# model = model.to(DEVICE)
nb_params = sum(p.numel() for p in model.parameters())
print(f"{nb_params:,}")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [4]:
# Get the full English benchmark
# benchmark = mteb.get_benchmark("MTEB(Multilingual)")
benchmark = mteb.get_benchmark("MTEB(eng, v2)")

# Filter to only retrieval tasks
classification_tasks = mteb.filter_tasks(benchmark, task_types=["Classification"])
len(classification_tasks)

8

In [5]:
cache = mteb.ResultCache(cache_path="~/.cache/mteb")
results = mteb.evaluate(model, tasks=classification_tasks, cache=cache)


Evaluating tasks:   0%|          | 0/8 [00:00<?, ?it/s]

Dataset 'Banking77Classification' is superseded by 'Banking77Classification.v2'. We recommend using the newer version of the dataset unless you are running a specific benchmark. See `get_task('Banking77Classification.v2').metadata.description` to get a description of the task and changes.
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/mteb/abstasks/abstask.py:122: UserWarning: Dataset 'Banking77Classification' is superseded by 'Banking77Classification.v2'. We recommend using the newer version of the dataset unless you are running a specific benchmark. See `get_task('Banking77Classification.v2').metadata.description` to get a description of the task and changes.
  warnings.warn(msg)
/mnt/data_1/aodonnat/miniforge3/envs/waldo/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarn

KeyboardInterrupt: 

In [18]:
import mteb

names = ["Multilingual", "Europe", "Indic", "eng, v2"]
comp = 0
for benchmark_name in names:
    # Get the full English benchmark
    benchmark = mteb.get_benchmark(f"MTEB({benchmark_name})")

    # Filter to only retrieval tasks
    classification_tasks = mteb.filter_tasks(benchmark, task_types=["Classification"])
    comp += len(classification_tasks)
    print(f"Found {len(classification_tasks)} classification tasks")
print(comp)


Found 43 classification tasks
Found 20 classification tasks
Found 12 classification tasks
Found 8 classification tasks
83
